In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.nn import functional as F

from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt

import numpy as np

In [2]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

Device: cpu


In [3]:
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

### Przygotowanie danych

In [4]:
import random


class SinogramNoise:
    def __init__(self, mean=0.0, std=0.001, p=1.0):
        self.mean = mean
        self.std = std
        self.p = p

    def __call__(self, sample):
        sinogram, image = sample

        if random.random() < self.p:
            noise = torch.randn_like(sinogram) * self.std + self.mean
            sinogram = sinogram + noise
            sinogram.clamp_(0.0, 1.0)

        return sinogram, image

In [6]:
from dival import get_standard_dataset

dataset = get_standard_dataset(
    "custom",
    data_path="../data/ct_ellipses_dataset",
    sinogram_shape=(256, 183),
    image_shape=(128, 128),
    parts_len={"train": 80, "validation": 10, "test": 10},
    impl="skimage",
)

transform_train = SinogramNoise(mean=0.0, std=0.0, p=1.0)
transform_test = SinogramNoise(mean=0.0, std=0.0, p=1.0)

train_dataset = dataset.create_torch_dataset(part="train", transform=transform_train)
test_dataset = dataset.create_torch_dataset(part="test", transform=transform_test)
val_dataset = dataset.create_torch_dataset(part="validation", transform=transform_test)

In [7]:
batch_size = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

#### Obliczenie odchylenia standardowego szumu przy augmentacji sinogramów

In [ ]:
from tqdm import tqdm

@torch.no_grad()
def compute_dataset_signal_power(loader):
    total_power = 0.0
    total_count = 0

    for sinograms, _ in tqdm(loader, desc="Computing signal power", unit="batch"):
        # sinograms: [B, ...]
        total_power += sinograms.pow(2).sum().item()
        total_count += sinograms.numel()

    return total_power / total_count


def std_from_dataset(loader, target_snr_db):
    signal_power = compute_dataset_signal_power(loader)
    std = (signal_power / (10 ** (target_snr_db / 10))) ** 0.5
    return std

In [ ]:
TARGET_SNR_DB = 45.0

global_std = std_from_dataset(train_loader, TARGET_SNR_DB)
transform_train.std = global_std

print("Noise transform std:", global_std)

### Model Pix2Pix

In [8]:
from models.Pix2Pix_128 import UnetGenerator, ConditionalDiscriminator

In [9]:
class GeneratorLoss(nn.Module):
    def __init__(self, alpha=100, beta=100):
        super().__init__()

        self.alpha = alpha
        self.beta = beta

        self.bce = nn.BCEWithLogitsLoss()
        self.l1 = nn.L1Loss()
        self.mse = nn.MSELoss()

    def forward(self, fake, real, fake_pred):
        fake_target = torch.ones_like(fake_pred)
        loss = (
            self.bce(fake_pred, fake_target)
            + self.alpha * self.l1(fake, real)
            + self.beta * self.mse(fake, real)
        )
        return loss


class DiscriminatorLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.loss_fn = nn.BCEWithLogitsLoss()

    def forward(self, fake_pred, real_pred):
        fake_target = torch.zeros_like(fake_pred)
        real_target = torch.ones_like(real_pred)
        fake_loss = self.loss_fn(fake_pred, fake_target)
        real_loss = self.loss_fn(real_pred, real_target)
        loss = (fake_loss + real_loss) / 2
        return loss

### Trening modelu

In [10]:
generator = UnetGenerator().to(device)
discriminator = ConditionalDiscriminator().to(device)

g_optimizer = torch.optim.Adam(generator.parameters(), lr=0.001)
d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=0.001)

g_criterion = GeneratorLoss(alpha=100, beta=100)
d_criterion = DiscriminatorLoss()

Schedulery

In [11]:
num_epochs = 50
decay_start = 25


def lambda_rule(epoch):
    if epoch < decay_start:
        return 1.0
    else:
        return 1.0 - (epoch - decay_start) / (num_epochs - decay_start)


g_scheduler = torch.optim.lr_scheduler.LambdaLR(g_optimizer, lr_lambda=lambda_rule)
d_scheduler = torch.optim.lr_scheduler.LambdaLR(d_optimizer, lr_lambda=lambda_rule)

In [12]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Number of trainable parameters generator:", count_parameters(generator))
print("Number of trainable parameters discriminator:", count_parameters(discriminator))

Number of trainable parameters generator: 51053569
Number of trainable parameters discriminator: 2763713


Precyzja połówkowa - scalery

In [11]:
from torch.amp.grad_scaler import GradScaler
from torch.amp.autocast_mode import autocast

scaler_g = GradScaler(device="cuda")
scaler_d = GradScaler(device="cuda")

In [12]:
import os


def save_state(epoch, generator, discriminator, g_optimizer, d_optimizer):
    checkpoint_dir = "checkpoints"
    os.makedirs(checkpoint_dir, exist_ok=True)

    checkpoint = {
        "epoch": epoch,
        "generator_state_dict": generator.state_dict(),
        "discriminator_state_dict": discriminator.state_dict(),
        "g_optimizer_state_dict": g_optimizer.state_dict(),
        "d_optimizer_state_dict": d_optimizer.state_dict(),
    }
    torch.save(checkpoint, f"{checkpoint_dir}/checkpoint_{epoch}.pt")

In [ ]:
import math


def calculate_psnr(img1, img2, max_val=1.0):
    img1_np = img1.squeeze().cpu().numpy()
    img2_np = img2.squeeze().cpu().numpy()
    mse = np.mean((img1_np - img2_np) ** 2)
    if mse == 0:
        return float("inf")
    return 20 * math.log10(max_val) - 10 * math.log10(mse)


def calculate_mse(img1, img2):
    return F.mse_loss(img1, img2).item()


def calculate_ssim(img1, img2):
    img1_np = img1.squeeze().cpu().numpy()
    img2_np = img2.squeeze().cpu().numpy()
    ssim_value = ssim(img1_np, img2_np, data_range=img1_np.max() - img1_np.min())
    # Ensure we return only the scalar score.
    if isinstance(ssim_value, (tuple, list)):
        ssim_value = ssim_value[0]
    return float(ssim_value)

In [ ]:
patience = num_epochs

best_weights_g = None
best_weights_d = None

best_loss = float("inf")
early_stopping_counter = 0

train_losses = []
mse_val_losses = []
ssim_val_losses = []
psnr_val_losses = []

print("Training started")

for epoch in range(num_epochs):
    ge_loss = 0.0
    de_loss = 0.0

    generator.train()
    discriminator.train()

    # Training loop with progress bar
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)
    for sino, img in train_bar:
        sino = sino.unsqueeze(1).to(device, non_blocking=True)  # [B, 1, H, W]
        img = img.unsqueeze(1).to(device, non_blocking=True)  # [B, 1, H, W]

        # Generator`s loss
        with autocast(device_type="cuda"):
            fake = generator(sino)
            fake_pred = discriminator(fake, sino)
            g_loss = g_criterion(fake, img, fake_pred)

        # Discriminator`s loss
        with autocast(device_type="cuda"):
            fake = generator(sino).detach()
            fake_pred = discriminator(fake, sino)
            real_pred = discriminator(img, sino)
            d_loss = d_criterion(fake_pred, real_pred)

        # Generator`s params update
        g_optimizer.zero_grad()
        scaler_g.scale(g_loss).backward()
        scaler_g.step(g_optimizer)
        scaler_g.update()

        # Discriminator`s params update
        d_optimizer.zero_grad()
        scaler_d.scale(d_loss).backward()
        scaler_d.step(d_optimizer)
        scaler_d.update()

        ge_loss += g_loss.item()
        de_loss += d_loss.item()

    # Schedulers step every epoch
    g_scheduler.step()
    d_scheduler.step()

    ge_loss /= len(train_loader)
    de_loss /= len(train_loader)

    # Validation loop with progress bar
    generator.eval()
    running_loss = ge_loss + de_loss
    mse_test_loss = 0.0
    ssim_val_loss = 0.0
    psnr_val_loss = 0.0
    with torch.no_grad():
        with autocast(device_type="cuda"):
            val_bar = tqdm(
                val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]", leave=False
            )
            for sino, img in val_bar:
                sino = sino.unsqueeze(1).to(device, non_blocking=True)
                img = img.unsqueeze(1).to(device, non_blocking=True)

                # MSE val loss
                output = generator(sino)
                mse_test_loss += calculate_mse(output, img)

                # SSIM val loss
                ssim_val_loss += calculate_ssim(output, img)

                # PSNR val loss
                psnr_val_loss += calculate_psnr(output, img)

    mse_test_loss /= len(val_loader)
    ssim_val_loss /= len(val_loader)
    psnr_val_loss /= len(val_loader)

    # Save checkpoints
    if epoch % 5 == 0:
        save_state(epoch, generator, discriminator, g_optimizer, d_optimizer)

    # Check if this is the best model so far in terms of validation mse loss
    if mse_test_loss < best_loss:
        best_loss = mse_test_loss
        best_weights_g = generator.state_dict()
        best_weights_d = discriminator.state_dict()

    # Early stopping
    if mse_test_loss > (1.01 * mse_val_losses[-1] if mse_val_losses else float("inf")):
        early_stopping_counter += 1

    if early_stopping_counter >= patience:
        print("Early stopping triggered")
        break

    train_losses.append(running_loss)
    mse_val_losses.append(mse_test_loss)
    ssim_val_losses.append(ssim_val_loss)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {running_loss:.4f} | "
        f"Val MSE: {mse_test_loss:.6f} | "
        f"Val SSIM: {ssim_val_loss:.4f} | "
        f"Val PSNR: {psnr_val_loss:.4f}"
    )

print("Training completed")

Training started


Epoch 1/50:   1%|▏         | 163/12884 [01:31<1:29:43,  2.36it/s]

### Testowanie, zapisanie wag

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(train_losses) + 1), train_losses, label="Train Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training Loss over Epochs")
plt.legend()
plt.show()

# Plot validation MSE
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(mse_val_losses) + 1), mse_val_losses, label="Validation MSE")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Validation MSE over Epochs")
plt.legend()
plt.show()

# Plot validation SSIM
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(ssim_val_losses) + 1), ssim_val_losses, label="Validation SSIM")
plt.xlabel("Epochs")
plt.ylabel("SSIM")
plt.title("Validation SSIM over Epochs")
plt.legend()
plt.show()

# Plot validation PSNR
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(psnr_val_losses) + 1), psnr_val_losses, label="Validation PSNR")
plt.xlabel("Epochs")
plt.ylabel("PSNR")
plt.title("Validation PSNR over Epochs")
plt.legend()
plt.show()

In [ ]:
# Test loop with progress bar
generator.eval()
mse_test_loss = 0.0
ssim_test_loss = 0.0
psnr_test_loss = 0.0
with torch.no_grad():
    with autocast(device_type="cuda"):
        val_bar = tqdm(
            test_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Test]", leave=False
        )
        for sino, img in val_bar:
            sino = sino.unsqueeze(1).to(device, non_blocking=True)
            img = img.unsqueeze(1).to(device, non_blocking=True)

            # MSE val loss
            output = generator(sino)
            mse_test_loss += calculate_mse(output, img)

            # SSIM val loss
            ssim_test_loss += calculate_ssim(output, img)

            # PSNR val loss
            psnr_test_loss += calculate_psnr(output, img)

mse_test_loss /= len(test_loader)
ssim_test_loss /= len(test_loader)
psnr_test_loss /= len(test_loader)

In [ ]:
# Save generator with best weights
generator.load_state_dict(best_weights_g)
torch.save(generator.state_dict(), "best_generator.pth")